# Milestone 3 — Retrieval-Augmented Generation (RAG)

Dataset: [Smart MCQ Solver Challenge](https://www.kaggle.com/competitions/smart-mcq-solver-challenge)

In [2]:
!pip install faiss-cpu --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 63.3 MB/s eta 0:00:00:00:0100:01


## Setup — Build the Knowledge Base + FAISS Index

The knowledge base (`kb`) is simply the text of the *correct* option for every row in `train.csv`.

In [3]:
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Knowledge base = the correct answer text for every row
kb = [str(row[row["answer"]]) for _, row in train.iterrows()]

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embed_model.encode(kb, show_progress_bar=False)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base size:", len(kb))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base size: 2000


## Q1 — Zero-shot classifier probability for the correct option (Row 150)

Run `facebook/bart-large-mnli` zero-shot classification on the prompt, using all 5 options as candidate labels.

In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150[opt]) for opt in ["A", "B", "C", "D", "E"]]
correct_text_150 = str(row_150[row_150["answer"]])

result_150 = zs(prompt_150, labels_150)

correct_score = result_150["scores"][result_150["labels"].index(correct_text_150)]
print("Probability of correct option:", round(correct_score, 3))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Probability of correct option: 0.384


## Q2 — FAISS retrieval rank of the true document (Row 150)

A **bi-encoder** (like MiniLM here) embeds the question and every document *separately*, then compares them by distance. It's fast but can miss context — that's why the true document doesn't always come back at rank 1.

In [5]:
prompt_150_embedding = embed_model.encode([prompt_150])
_, retrieved_indices = index.search(prompt_150_embedding, 10)
retrieved_indices = retrieved_indices[0]

print("Retrieved KB indices:", retrieved_indices)

rank = list(retrieved_indices).index(150) + 1 if 150 in retrieved_indices else "Not in top 10"
print("Rank of true document:", rank)

Retrieved KB indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Rank of true document: 10


## Q3 — Cross-Encoder reranking

A **cross-encoder** feeds the question and document into the model *together*, so attention can directly compare their words — slower than a bi-encoder, but far more accurate. We rerank the same 10 FAISS results.

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

ranked_order = sorted(range(len(docs_10)), key=lambda i: ce_scores[i], reverse=True)
ranked_kb_indices = [retrieved_indices[i] for i in ranked_order]

rerank_position = list(ranked_kb_indices).index(150) + 1
print("Cross-encoder rank of true document:", rerank_position)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder rank of true document: 1


## Q4 — Token count of a retrieved-context prompt (Row 42)

Retrieve the top 5 documents, build a `"Context: ... Question: ..."` string, and count tokens with the BERT tokenizer (no truncation).

In [7]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompt_42 = str(train.iloc[42]["prompt"])
prompt_42_embedding = embed_model.encode([prompt_42])
_, idx_42 = index.search(prompt_42_embedding, 5)
docs_42 = [kb[i] for i in idx_42[0]]

rag_string_42 = "Context: " + " ".join(docs_42) + " Question: " + prompt_42

token_count = len(bert_tokenizer(rag_string_42, truncation=False)["input_ids"])
print("Total tokens:", token_count)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens: 216


## Q5 — Zero-shot classification with the TRUE document as context (Row 150)

Now give the classifier the correct context directly — the exact document at KB index 150 — and see how much the correct option's probability improves.

In [8]:
true_doc_150 = kb[150]
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_rag = zs(rag_string_150, labels_150)
correct_score_rag = result_rag["scores"][result_rag["labels"].index(correct_text_150)]

print("New probability of correct option (true context):", round(correct_score_rag, 3))

New probability of correct option (true context): 0.989


## Q6 — Adversarial RAG: what if retrieval gives the WRONG context?

**Garbage in, garbage out.** Force the context to an unrelated document (KB index 999) and see how badly the correct option's probability drops.

In [9]:
adversarial_doc = kb[999]
adversarial_string = f"Context: {adversarial_doc} Question: {prompt_150}"

result_adv = zs(adversarial_string, labels_150)
correct_score_adv = result_adv["scores"][result_adv["labels"].index(correct_text_150)]

print("Probability of correct option (adversarial context):", round(correct_score_adv, 3))

Probability of correct option (adversarial context): 0.529


## Q7 — Hit Rate over the first 100 rows

A **"hit"** = the correct option's exact text appears somewhere inside the top-5 retrieved documents for that row.

In [10]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    correct_text = str(row[row["answer"]])

    prompt_embedding = embed_model.encode([str(row["prompt"])])
    _, retrieved = index.search(prompt_embedding, 5)
    retrieved_docs = [kb[j] for j in retrieved[0]]

    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

hit_rate = (hits / 100) * 100
print(f"Hit Rate: {hit_rate:.1f}%")

Hit Rate: 73.0%


## Q8 — Full RAG Pipeline: Retrieve → Rerank → Augment → Predict (first 20 rows)

The complete pipeline, per row:
1. **Retrieve** top-5 documents with FAISS
2. **Rerank** with the cross-encoder, keep the single best document
3. **Augment** the prompt with that document as context
4. **Predict** with zero-shot classification over all 5 options, take the top-3, score with MAP@3

In [11]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    for idx, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (idx + 1)
    return 0.0

options = ["A", "B", "C", "D", "E"]
scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_text = str(row["prompt"])
    option_labels = [str(row[opt]) for opt in options]
    actual_letter = row["answer"]

    # Retrieve
    prompt_embedding = embed_model.encode([prompt_text])
    _, retrieved = index.search(prompt_embedding, 5)
    docs_5 = [kb[j] for j in retrieved[0]]

    # Rerank -> keep best document
    pairs = [[prompt_text, doc] for doc in docs_5]
    ce_scores_i = cross_encoder.predict(pairs)
    best_doc = docs_5[ce_scores_i.argmax()]

    # Augment
    rag_string = f"Context: {best_doc} Question: {prompt_text}"

    # Predict
    result = zs(rag_string, option_labels)
    label_to_letter = dict(zip(option_labels, options))
    top3_letters = [label_to_letter[label] for label in result["labels"][:3]]

    scores.append(apk(actual_letter, top3_letters))

final_map3 = sum(scores) / len(scores)
print(f"RAG Pipeline MAP@3: {final_map3:.3f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


RAG Pipeline MAP@3: 0.975
